# 05 — Idiot Sandwich: TRL DPOTrainer + LoRA persona tuning (Task 5)

Accelerator: **GPU T4 x2** (uses one GPU). Model `HuggingFaceTB/SmolLM2-360M-Instruct`, data
`data/persona_lora/ramsay_style_preferences.json` (220 rows; 22 held out). LoRA and DPO settings are in
`configs/lora_dpo.yaml` and must be justified in the report.

In [ ]:
# Clone (or update) the project repo into /kaggle/working. Notebook settings: Internet ON.
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")

In [ ]:
# Manual section 5.2 plus PEFT/TRL for the LoRA task. torchao is removed to avoid the PEFT conflict.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11.0" "trl>=0.24,<0.30"
!python -m pip install -q -e . --no-deps

In [ ]:
# Manual section 7: confirm two T4 GPUs and print package versions.
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
sh("python scripts/train_lora_dpo.py --config configs/lora_dpo.yaml")

In [ ]:
!cat outputs/lora_dpo/summary.json
!python scripts/plot_metrics.py --run lora_dpo=outputs/lora_dpo --out_dir outputs/plots
from IPython.display import Image, display
display(Image("outputs/plots/lora_dpo_metrics.png"))

In [ ]:
# Manual section 5.5: archive the run for easy attaching/downloading, then use "Save Version".
!cd outputs && zip -q -r -0 /kaggle/working/lora_dpo_run_outputs.zip lora_dpo
!ls -lh /kaggle/working/*.zip
!du -sh outputs/lora_dpo